### Considere la familia de funciones

### $f_k(x) = -\frac{x^2}{10k^2+1} + (x-2)^{3/2} +10$

### Queremos estudiar para qué valores de $k>0$ la misma admite un punto fijo $p_0$ donde el método de Punto Fijo converja. Queremos estimar dichos $k$ 's con la mayor precisión posible. La idea es que una vez obtenidos los mismos, podríamos graficar la evolución de los puntos fijos con muy pocas iteraciones utilizando algún método acelerado. ¿Será que esto es mejor que aplicar otro método de forma directa? Estudiar este problema pondrá en juego nuestros conocimientos sobre métodos numéricos para resolver ecuaciones no lineales. En esta ocasión nos focalizaremos sólo en calcular los $k$`s mencionados al comienzo.

### Se pide que encuentre el intervalo óptimo de $k$ 's teniendo en cuenta una tolerancia de $10^{-3}$ de forma tal que la sucesión de Punto Fijo converja.

### CONDICIONES DE RESOLUCIÓN:
1. Justifique con GeoGebra y desde la teoría la estrategia para resolver el problema.
2. Escriba el contrato en TODAS las funciones que defina.
3. No deje "código suelto". Defina funciones que realizan lo que desea.
4. Haga comentarios apropiados que permitan seguir el hilo de su razonamiento.
5. Su código debe ser acorde a lo visto en clase. Evite funciones *mágicas* o *extrañas* generadas por IA.
6. Su código debe seguir las buenas prácticas.

### Sugerencias:
1. Defina todas sus funciones escogiendo como primer variable `k`. Por ejemplo, `f(k, x)`. Así se logra una forma más afín a la notación matemática $f_k(x)$.
2. Nombres de algunas funciones posibles en su resolución `f`, `f2`, `df`, `secante`, `convergencia_pf`, `buscador_convergencia_pf`, `punto_fijo`.

In [1]:
from math import sqrt

def f(k,x):
  return - (x**2)/(10*k**2+1) + (x-2)**(3/2) +10

# Comparar con GeoGebra
print(f(1,2), f(1,3), f(1,116))

# Sacado de clase codeo inteligente 2025 adaptado a 2 variables
def secante(k, h, intervalo, n):
    """
    PROPÓSITO: calcula el término *n* de la sucesión de la Secante aplicada a *h* tomando como valores iniciales los extremos de *intervalo*.

    PRECONDICIONES: 
        - h debe estar definida en el entorno de los valores iniciales.
        - intervalo debe ser una lista de dos números distintos.
        - n debe ser un entero positivo.

    PARÁMETROS:
        - h. función. La función involucrada.
        - intervalo. Lista de dos números. Valores iniciales para comenzar las iteraciones.
        - n. Entero positivo. El número de iteraciones a realizar.
    """
    x0, x1 = intervalo
    y0, y1 = h(k, x0), h(k, x1)
    i = 0
    # como una resta cuesta poco, en este código podría ser lazy y no optimizar
    while i < n and (y1-y0) != 0:
        #print(x0, y0)
        p = x1 - y1 * (x1 - x0) / (y1 - y0)
        x0, y0 = x1, y1
        x1, y1 = p, h(k, p)
        i = i + 1
    return k, p

# Ecuación de PF como raíz de h
def f2(k, x):
  return (x**2)/(10*k**2+1) - (x-2)**(3/2) + x -10

def df(k, x):
  return 1.5*sqrt(x-2) -(2*x)/(10*k**2+1)
  

9.636363636363637 10.181818181818182 3.914193458842192


In [2]:
f2(0.825, 39)

-1.2183385209540347

### Preparativos para buscar el borde derecho

In [12]:
# simplificamos la función sabiendo que sólo vale para f2
def convergencia_pf(k, intervalo):
  """
  
  """
  x1, x2 = intervalo
  k_punto_fijo, punto_fijo = secante(k,f2, [x1, x2], 6)
  return k, punto_fijo, df(k_punto_fijo, punto_fijo)

print(convergencia_pf(0.825, [39, 42] ))
print(convergencia_pf(0.845, [39, 42] ))

(0.825, 39.63775906122246, -0.9529475930857316)
(0.845, 44.17165952344663, -1.1116977652548758)


In [28]:
import numpy as np

def buscador_convergencia_pf(intervalo_k, intervalo_busqueda_pf, TOL):
  """
  """
  x1, x2 = intervalo_busqueda_pf
  k0, k1 = intervalo_k
  samples = np.linspace(k0, k1)
  i = 1
  ro =1
  while i<len(samples) and ro > TOL:
    i = i+1
    punto_fijo_k = convergencia_pf(samples[i], [x1,x2])[2]
    ro = abs(abs(punto_fijo_k) -1)
    print(i, samples[i], punto_fijo_k, ro)
  return samples[i], punto_fijo_k, ro

# Arranqué probando [0.825, 0.845] basado en GeoGebra, luego mirando los prints fui acortando.
buscador_convergencia_pf([0.831, 0.83118],[39,42], 1e-3)


2 0.8310073469387754 -0.9997070678236657 0.000292932176334304


(np.float64(0.8310073469387754),
 np.float64(-0.9997070678236657),
 np.float64(0.000292932176334304))

### Ahora busquemos el borde izquierdo

In [46]:
# Comencé con [0.425, 0.45] para k basado den GeoGebra 
buscador_convergencia_pf([0.4470812, 0.4485],[4,6], 1e-4)

2 0.44713911020408165 -1.0004997110969578 0.0004997110969577889
3 0.44716806530612246 -1.0003054439633248 0.00030544396332476964
4 0.4471970204081633 -1.0001111914998413 0.00011119149984128995
5 0.4472259755102041 -0.9999169537076629 8.304629233713001e-05


(np.float64(0.4472259755102041),
 np.float64(-0.9999169537076629),
 np.float64(8.304629233713001e-05))

### Ahora grafiquemos usando PF (la gracia es que se podría luego acelerar la convergencia con Stefenssen)

In [55]:
k_inf, k_sup = buscador_convergencia_pf([0.447, 0.45],[4,6], 1e-3)[0], buscador_convergencia_pf([0.831, 0.83118],[39,42], 1e-3)[0]

def punto_fijo(k, h, x0, n):
    """
    PROPÓSITO: calcula el término *n* de la sucesión de Punto Fijo comenzando en *x0* para *h*.

    PRECONDICIONES: 
    - h debe estar definida en el entorno de x0. 
    - n debe ser un entero positivo.

    PARÁMETROS:
    h  -- función de iteración.
    x0 -- Número. Valor inicial para comenzar las iteraciones.
    n  -- Entero positivo. El número de iteraciones a realizar.
    """
    p = x0
    for i in range(n):
        p = h(k,p)
    return i, p

punto_fijo(0.625, f,  11, 15)

2 0.44712244897959186 -1.0006115021326303 0.0006115021326302816
2 0.8310073469387754 -0.9997070678236657 0.000292932176334304


(14, 12.233211465399368)